In [ ]:
# ===== EXPORT RISK SCORE =====
# Prediksi untuk SEMUA eligible customer (bukan hanya test set)
X_all = model_data.drop(columns=['customer_unique_id', 'is_repeat'])
X_all = pd.get_dummies(X_all, columns=['customer_state'], drop_first=True)

# Align kolom dengan training (kalau ada kolom baru/hilang)
X_all = X_all.reindex(columns=X_train.columns, fill_value=0)

# Prediksi probability
risk_scores = xgb_smote.predict_proba(X_all)[:, 1]

# Gabung dengan customer info
output = model_data[['customer_unique_id', 'is_repeat']].copy()
output['risk_score'] = risk_scores
output['risk_segment'] = pd.cut(
    output['risk_score'],
    bins=[0, 0.1, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)
output = output.sort_values('risk_score', ascending=False)

# Simpan
output.to_csv('../data/processed/customer_risk_scores.csv', index=False)
print(f"✅ Exported: {len(output):,} customers dengan risk score")
print(f"\n📊 Distribusi segment:")
print(output['risk_segment'].value_counts().sort_index())
print(f"\n📊 Top 5 customer paling berisiko:")
print(output.head()[['customer_unique_id', 'risk_score', 'risk_segment']])